In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import requests
import warnings
from sklearn.decomposition import PCA

import plotly.express as px 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import euclidean_distances
from scipy.spatial.distance import cdist


load_dotenv()
SPOTIPY_CLIENT_ID = os.getenv("SPOTIPY_CLIENT_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIPY_CLIENT_SECRET")
SPOTIPY_REDIRECT_URL = os.getenv("SPOTIPY_REDIRECT_URL")

In [ ]:
scope = "user-library-read"
auth_manager = SpotifyOAuth(
    SPOTIPY_CLIENT_ID, 
    SPOTIPY_CLIENT_SECRET, 
    redirect_uri=SPOTIPY_REDIRECT_URL, 
    scope=scope
)

sp = spotipy.Spotify(auth_manager=auth_manager)

In [ ]:
pages = 50
limit = 50
offset = 0
track_list = []

for i in range(pages):
    results = sp.current_user_top_tracks(limit=limit, offset=offset + limit)
    for idx, item in enumerate(results['items']):
        track_list.append(item)
    offset += limit
        # track = item['track']
        # print(idx, track['artists'][0]['name'], " – ", track['name'])

In [ ]:
track_df = pd.DataFrame(track_list)
num_tracks = len(track_df)
divisions = num_tracks // 40
remainder = num_tracks % 40
print(num_tracks)
print(divisions)

## Get track data from ReccoBeats

In [ ]:
div = 0
feature_list = []
track_list = []
headers = {
    'Accept': 'application/json'
}

for i in range(divisions):
    if i == divisions - 1:
        track_ids = track_df['id'][div: div + remainder]
    else:
        track_ids = track_df['id'][div: div + 40]
    
    id_string = ",".join(track_ids)

    feature_url = f"https://api.reccobeats.com/v1/audio-features?ids={id_string}"
    track_url = f"https://api.reccobeats.com/v1/track?ids={id_string}"
    response = requests.request("GET", feature_url, headers=headers)
    features = response.json()["content"]

    for feature in features:
        feature_list.append(feature)

    response = requests.request("GET", track_url, headers=headers)
    tracks = response.json()["content"]

    for track in tracks:
        track_list.append(track)
    
    
    div += 40

In [ ]:
feature_df = pd.DataFrame(feature_list)
feature_df['id'] = feature_df['href'].str.removeprefix("https://open.spotify.com/track/")

recco_df = pd.DataFrame(track_list)
recco_df['id'] = recco_df['href'].str.removeprefix("https://open.spotify.com/track/")

feature_df = pd.merge(feature_df, recco_df, on='id', how='inner')

In [ ]:
df = pd.merge(track_df, feature_df, on='id', how='inner')

In [ ]:
df['rankings'] = [-(i - len(df) + 1) / (len(df) - 1) for i in range(len(df))]
df['album'] = [album.get("name") for album in df.album]

In [ ]:
song_cluster_pipeline = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('kmeans', KMeans(n_clusters=4, verbose=False))
    ], 
    verbose=False
)
X = df[['duration_ms', 'acousticness', 'danceability', 'energy', 'instrumentalness', 
        'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'rankings', 'popularity']]
number_cols = list(X.columns)
song_cluster_pipeline.fit(X)
song_cluster_labels = song_cluster_pipeline.predict(X)
df['cluster_label'] = song_cluster_labels

warnings.filterwarnings('ignore')

In [ ]:
pca_pipeline = Pipeline([('scaler', StandardScaler()), ('PCA', PCA(n_components=2))])
song_embedding = pca_pipeline.fit_transform(X)
projection = pd.DataFrame(columns=['x', 'y'], data=song_embedding)
projection['title'] = df['name']
projection['album'] = df['album']
projection['cluster'] = df['cluster_label'].astype("int")

fig = px.scatter(
    projection, x='x', y='y', color='cluster', hover_data=['title', 'album'])
fig.show()